In [7]:
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd 
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


In [ ]:
root = Path("C:/Users/mlady/Documents/202620/stat6685/finalproject/mallorn-astronomical-classification-challenge/data")

# 1) Global training metadata with labels
train_log = pd.read_csv(root / "train_log.csv")
# expected columns include: object_id, split, target, Z, EBV, ...
print("train_log shape:", train_log.shape)
print(train_log[["split", "target"]].head())

# 2) Load one split's lightcurves and attach labels/features from global log
def load_train_split(split_name: str):
    lc_path = root / split_name / "train_full_lightcurves.csv"
    lc = pd.read_csv(lc_path)

    # Keep only metadata rows for this split
    meta = train_log.loc[train_log["split"] == split_name].copy()

    # Merge target + static features onto each observation row
    merged = lc.merge(
        meta[["object_id", "target", "Z", "Z_err", "EBV", "SpecType", "split"]],
        on="object_id",
        how="left",
        validate="many_to_one"
    )
    return merged, meta

# Example
train_s1, meta_s1 = load_train_split("split_01")
print("split_01 observations:", train_s1.shape)
print("split_01 objects:", meta_s1["object_id"].nunique())
print(train_s1[["object_id", "Time (MJD)", "Flux", "Filter", "target"]].head())

train_log shape: (3043, 8)
      split  target
0  split_01       0
1  split_01       0
2  split_01       0
3  split_01       0
4  split_01       0
split_01 observations: (26324, 11)
split_01 objects: 155
                  object_id  Time (MJD)       Flux Filter  target
0  Dornhoth_fervain_onodrim  63314.4662  -1.630159      z       0
1  Dornhoth_fervain_onodrim  63780.9674  10.499389      r       0
2  Dornhoth_fervain_onodrim  63789.7693   5.866250      y       0
3  Dornhoth_fervain_onodrim  63794.1702   3.903623      r       0
4  Dornhoth_fervain_onodrim  63794.1702   5.226644      i       0


In [ ]:

"""DATA LOADING"""
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]
FILTER_TO_IDX = {f: i for i, f in enumerate(FILTER_ORDER)}

def encode_sequence(obj_df):
    obj_df = obj_df.sort_values("Time (MJD)").copy()
    t = obj_df["Time (MJD)"].to_numpy(dtype=np.float32)
    flux = obj_df["Flux"].to_numpy(dtype=np.float32)
    flux_err = obj_df["Flux_err"].to_numpy(dtype=np.float32)

    dt = np.diff(t, prepend=t[0]).astype(np.float32)

    filt_oh = np.zeros((len(obj_df), len(FILTER_ORDER)), dtype=np.float32)
    filt_idx = obj_df["Filter"].map(FILTER_TO_IDX).to_numpy()
    valid = ~pd.isna(filt_idx)
    filt_oh[np.arange(len(obj_df))[valid], filt_idx[valid].astype(int)] = 1.0

    x = np.column_stack([flux, flux_err, dt, filt_oh]).astype(np.float32)
    return torch.tensor(x, dtype=torch.float32)

class LightCurveSplitDataset(Dataset):
    def __init__(self, split_name, log_df, root_path, is_test=False):
        self.split_name = split_name
        self.train_log_df = log_df
        self.root_path = root_path
        self.is_test = is_test

        lc_path = root_path / split_name / f"{'test' if is_test else 'train'}_full_lightcurves.csv"
        self.lc = pd.read_csv(lc_path)

        self.meta = log_df.loc[log_df["split"] == split_name].copy()
        self.meta = self.meta.set_index("object_id", drop=False)

        object_set = set(self.lc["object_id"].unique())
        self.object_ids = [oid for oid in self.meta["object_id"].tolist() if oid in object_set]
        self.grouped = dict(tuple(self.lc.groupby("object_id")))

    def __len__(self):
        return len(self.object_ids)

    def __getitem__(self, idx):
        oid = self.object_ids[idx]
        obj_df = self.grouped[oid]
        m = self.meta.loc[oid]

        seq = encode_sequence(obj_df)

        z = 0.0 if pd.isna(m.get("Z", np.nan)) else float(m["Z"])
        z_err = 0.0 if pd.isna(m.get("Z_err", np.nan)) else float(m["Z_err"])
        ebv = 0.0 if pd.isna(m.get("EBV", np.nan)) else float(m["EBV"])
        static = torch.tensor([z, z_err, ebv], dtype=torch.float32)

        y = torch.tensor(float(m["target"]), dtype=torch.float32)

        return {
            "object_id": oid,
            "seq": seq,
            "static": static,
            "y": y
        }

"""
Build one mini-batch from a list of dataset samples.

This function:
- collects variable-length sequence tensors (`seq`)
- records each sequence length (`lengths`)
- pads sequences to the max length in the batch (`seq_padded`)
- stacks fixed-size static features (`static`) and labels (`y`)
- keeps object IDs as a Python list (`object_id`)

Returned shapes:
- seq: [B, T_max, F]
- lengths: [B]
- static: [B, 3]
- y: [B]
"""
def collate_pad(batch):
    seqs = [b["seq"] for b in batch]
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    seq_padded = pad_sequence(seqs, batch_first=True)

    static = torch.stack([b["static"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0)
    obj_ids = [b["object_id"] for b in batch]

    return {
        "object_id": obj_ids,
        "seq": seq_padded,
        "lengths": lengths,
        "static": static,
        "y": y
    }

# Example usage:
train_ds = LightCurveSplitDataset("split_01", train_log, root)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_pad)

batch = next(iter(train_loader))
print("seq shape:", batch["seq"].shape)        # [B, T_max, F]
print("lengths:", batch["lengths"][:8])
print("static shape:", batch["static"].shape)  # [B, 3]
print("y shape:", batch["y"].shape)            # [B]

seq shape: torch.Size([32, 939, 9])
lengths: tensor([126, 157, 127, 167, 159, 172, 167, 143])
static shape: torch.Size([32, 3])
y shape: torch.Size([32])


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=5, hidden_size=50, num_layers=1, batch_first=True)
        self.linear = nn.Linear(50, 1)
    def forward(self, x):
        x, _ = self.lstm(x)
        x = self.linear(x)
        return x

In [ ]:
model = LSTMModel()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.MSELoss()
loader = train_loader
X_train = batch["seq"]
y_train = batch["y"]

n_epochs = 2000
for epoch in range(n_epochs):
    model.train()
    print(len(loader))
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Validation
    if epoch % 100 != 0:
        continue
    model.eval()
    with torch.no_grad():
        y_pred = model(X_train)
        train_rmse = np.sqrt(loss_fn(y_pred, y_train))
    print("Epoch %d: train RMSE %.4f" % (epoch, train_rmse))

5


ValueError: too many values to unpack (expected 2)